In [3]:
from dotenv import load_dotenv
load_dotenv()

True

### FAISS

In [6]:
from langchain_community.document_loaders import TextLoader

load=TextLoader('./data/prac.txt')
docs=load.load()
docs[0]

Document(metadata={'source': './data/prac.txt'}, page_content='Once there was a small town at the edge of a wide plain. The town was quiet, almost forgotten, and people there lived slow lives. Every morning the sun rose gently, and every evening it went down behind the same old trees. Nothing dramatic ever happened, and that was exactly how the people liked it.\n\nIn this town lived a young man named Arin. He was not special in any obvious way. He was not the strongest, not the smartest, and not the richest. He worked at a small repair shop where he fixed broken clocks, radios, and old machines that most people had already given up on. Arin liked broken things. They made sense to him. When something was broken, there was always a reason.\n\nArin had one habit that made him different from others. Every night, after closing the shop, he walked to the open field outside the town and sat there quietly. He did not bring a phone or a book. He just sat and looked at the sky. The stars felt ho

In [7]:
from langchain_community.embeddings import OllamaEmbeddings
embeddings=OllamaEmbeddings(model='llama3.1')
embeddings

C:\Users\moury\AppData\Local\Temp\ipykernel_34216\886054143.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings=OllamaEmbeddings(model='llama3.1')


OllamaEmbeddings(base_url='http://localhost:11434', model='llama3.1', embed_instruction='passage: ', query_instruction='query: ', mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None, show_progress=False, headers=None, model_kwargs=None)

In [25]:
from langchain_text_splitters import CharacterTextSplitter

splitter=CharacterTextSplitter(chunk_size=500,chunk_overlap=50)
docs=splitter.split_documents(docs)

In [10]:
from langchain_community.vectorstores import FAISS
db=FAISS.from_documents(docs,embedding=embeddings)

In [11]:
query="He just sat and looked at the sky. The stars felt honest to him. They did not pretend."
res=db.similarity_search(query)
docs[0].page_content

'Once there was a small town at the edge of a wide plain. The town was quiet, almost forgotten, and people there lived slow lives. Every morning the sun rose gently, and every evening it went down behind the same old trees. Nothing dramatic ever happened, and that was exactly how the people liked it.'

In [13]:
retriver=db.as_retriever()
retriver.invoke(query)[0].page_content

'Once there was a small town at the edge of a wide plain. The town was quiet, almost forgotten, and people there lived slow lives. Every morning the sun rose gently, and every evening it went down behind the same old trees. Nothing dramatic ever happened, and that was exactly how the people liked it.'

In [14]:
db.similarity_search_with_score(query)

[(Document(id='0a975bf4-f312-49ae-8bd3-fba50deafa0e', metadata={'source': './data/prac.txt'}, page_content='Once there was a small town at the edge of a wide plain. The town was quiet, almost forgotten, and people there lived slow lives. Every morning the sun rose gently, and every evening it went down behind the same old trees. Nothing dramatic ever happened, and that was exactly how the people liked it.'),
  np.float32(7339.7925)),
 (Document(id='3a1ee175-5078-4098-af4c-3c3899e04294', metadata={'source': './data/prac.txt'}, page_content='Arin had one habit that made him different from others. Every night, after closing the shop, he walked to the open field outside the town and sat there quietly. He did not bring a phone or a book. He just sat and looked at the sky. The stars felt honest to him. They did not pretend. They were just there, shining or not shining, without caring who noticed.'),
  np.float32(7524.8027)),
 (Document(id='116712c9-23ca-4b9c-aac8-2258c2a2a91a', metadata={'so

In [15]:
embed_vector=embeddings.embed_query(query)
db.similarity_search_by_vector(embed_vector)

[Document(id='0a975bf4-f312-49ae-8bd3-fba50deafa0e', metadata={'source': './data/prac.txt'}, page_content='Once there was a small town at the edge of a wide plain. The town was quiet, almost forgotten, and people there lived slow lives. Every morning the sun rose gently, and every evening it went down behind the same old trees. Nothing dramatic ever happened, and that was exactly how the people liked it.'),
 Document(id='3a1ee175-5078-4098-af4c-3c3899e04294', metadata={'source': './data/prac.txt'}, page_content='Arin had one habit that made him different from others. Every night, after closing the shop, he walked to the open field outside the town and sat there quietly. He did not bring a phone or a book. He just sat and looked at the sky. The stars felt honest to him. They did not pretend. They were just there, shining or not shining, without caring who noticed.'),
 Document(id='116712c9-23ca-4b9c-aac8-2258c2a2a91a', metadata={'source': './data/prac.txt'}, page_content='He took the st

In [16]:
db.save_local('faiss_index')

In [18]:
new_db=FAISS.load_local('faiss_index',embeddings=embeddings,allow_dangerous_deserialization=True)

In [20]:
doc=db.similarity_search(query)
doc

[Document(id='0a975bf4-f312-49ae-8bd3-fba50deafa0e', metadata={'source': './data/prac.txt'}, page_content='Once there was a small town at the edge of a wide plain. The town was quiet, almost forgotten, and people there lived slow lives. Every morning the sun rose gently, and every evening it went down behind the same old trees. Nothing dramatic ever happened, and that was exactly how the people liked it.'),
 Document(id='3a1ee175-5078-4098-af4c-3c3899e04294', metadata={'source': './data/prac.txt'}, page_content='Arin had one habit that made him different from others. Every night, after closing the shop, he walked to the open field outside the town and sat there quietly. He did not bring a phone or a book. He just sat and looked at the sky. The stars felt honest to him. They did not pretend. They were just there, shining or not shining, without caring who noticed.'),
 Document(id='116712c9-23ca-4b9c-aac8-2258c2a2a91a', metadata={'source': './data/prac.txt'}, page_content='He took the st

### ChromaDB

In [23]:
from langchain_chroma import Chroma

In [27]:
db=Chroma.from_documents(docs,embeddings)
db

In [28]:
query="Arin had one habit that made him different from others. Every night, after closing the shop, he walked to the open field outside the town and sat there quietly."
docs=db.similarity_search(query)
docs[0].page_content

'Arin had one habit that made him different from others. Every night, after closing the shop, he walked to the open field outside the town and sat there quietly. He did not bring a phone or a book. He just sat and looked at the sky. The stars felt honest to him. They did not pretend. They were just there, shining or not shining, without caring who noticed.'

In [29]:
db=Chroma.from_documents(docs,embeddings,persist_directory='./chroma_db')
db

In [33]:
db2=Chroma(persist_directory='./chroma_db',embedding_function=embeddings)

In [35]:
docs=db2.similarity_search(query)
docs

[Document(id='ce6fd752-6be7-48f9-979f-c92cff05629f', metadata={'source': './data/prac.txt'}, page_content='Arin had one habit that made him different from others. Every night, after closing the shop, he walked to the open field outside the town and sat there quietly. He did not bring a phone or a book. He just sat and looked at the sky. The stars felt honest to him. They did not pretend. They were just there, shining or not shining, without caring who noticed.'),
 Document(id='95b3b092-c628-406e-8694-44158c3382a2', metadata={'source': './data/prac.txt'}, page_content='One night, while sitting in the field, Arin noticed something strange. A light moved across the sky, not like a star and not like a plane. It slowed down, changed direction, and then disappeared. Arin did not panic. He simply observed. Curiosity had always been stronger than fear for him.'),
 Document(id='f7567771-0f21-424c-81c3-54103b1602c1', metadata={'source': './data/prac.txt'}, page_content='Once there was a small to

In [36]:
db.as_retriever()
retriver.invoke(query)[0].page_content

'Arin had one habit that made him different from others. Every night, after closing the shop, he walked to the open field outside the town and sat there quietly. He did not bring a phone or a book. He just sat and looked at the sky. The stars felt honest to him. They did not pretend. They were just there, shining or not shining, without caring who noticed.'